In [1]:
import pandas as pd
import numpy as np
from flaml import AutoML

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-15 10:21:59,204	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-07-15 10:21:59,347	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
DATA_FOLDER = "./"
df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_2c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
df = df[df["product_id"].isin(product_ids)]
df = df.sort_values(by=["date_id", "product_id"])
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


/tmp/ipykernel_59591/2984009778.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [3]:
df = df.drop(columns=["fecha", "periodo_min_producto", "periodo_max_producto", "periodo_min_customer", "periodo_max_customer"], errors="ignore")
# transformo todas las columnas object en categoricas
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("category")
df.describe()

,product_id,customer_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,sku_size,year,mes,...,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_2,prod_tn_rolling_mean_24_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_3,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_2,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_24_lag_1,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_24_lag_1_x_tn_rolling_mean_12_lag_3,target
count,67125.000000,67125.000000,44750.000000,67125.000000,67125.000000,67125.000000,32181.000000,67125.000000,67125.000000,67125.000000,...,2.056500e+04,1.886100e+04,2.056500e+04,3.877200e+04,1.886100e+04,3.680400e+04,1.886100e+04,3.680400e+04,1.886100e+04,62445.000000
mean,20479.289698,6667.666667,0.008894,74.170294,17.101440,16.724100,21.202147,447.860565,2018.112089,6.681475,...,4.618019e+03,4.851869e+03,4.665210e+03,4.519523e+03,4.800815e+03,4.571645e+03,4.638457e+03,4.733664e+03,4.682647e+03,16.957232
std,334.547874,4714.787452,0.093888,115.286339,67.551514,65.345131,60.536484,831.341003,0.813328,3.445673,...,4.854685e+04,4.798484e+04,4.859605e+04,4.565110e+04,4.699408e+04,4.585604e+04,4.862984e+04,4.778962e+04,4.876376e+04,66.435036
min,20001.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-13.666560,1.000000,2017.000000,1.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,20200.000000,0.000000,0.000000,3.000000,0.110910,0.110790,1.422400,89.000000,2017.000000,4.000000,...,9.533407e-02,1.082089e-01,9.697029e-02,9.019850e-02,1.106257e-01,9.226073e-02,9.754751e-02,8.907180e-02,9.801301e-02,0.115910
50%,20411.000000,10001.000000,0.000000,11.000000,1.143290,1.139340,6.120440,220.000000,2018.000000,7.000000,...,2.828777e+00,2.942908e+00,2.846635e+00,2.616434e+00,2.973132e+00,2.672556e+00,2.854394e+00,2.579931e+00,2.883994e+00,1.161910
75%,20730.000000,10002.000000,0.000000,118.000000,8.066350,7.969530,19.142031,475.000000,2019.000000,10.000000,...,9.340449e+01,1.055332e+02,9.443301e+01,9.727163e+01,1.085154e+02,9.885401e+01,9.475851e+01,9.604533e+01,9.485516e+01,8.063050
max,21276.000000,10002.000000,1.000000,705.000000,2153.267822,2030.201416,1562.024536,10000.000000,2019.000000,12.000000,...,1.562886e+06,1.518555e+06,1.576501e+06,1.515214e+06,1.461905e+06,1.515214e+06,1.574931e+06,1.618976e+06,1.562886e+06,2030.201416


In [4]:
from sklearn.model_selection import BaseCrossValidator
import numpy as np

class CustomTimeSeriesSplit(BaseCrossValidator):
    def __init__(self, n_splits=3, gap=1):
        self.n_splits = n_splits
        self.gap = gap

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        # Asegurar que X es DataFrame
        
        unique_dates = sorted(X["date_id"].unique(), reverse=True)
        
        for i in range(self.n_splits):
                
            test_date_id = unique_dates[i]
            train_date_id = test_date_id - self.gap - 1
            
            # Usar np.where para obtener posiciones enteras
            train_mask = X["date_id"] <= train_date_id
            test_mask = X["date_id"] == test_date_id
            
            train_idx = np.where(train_mask)[0]
            test_idx = np.where(test_mask)[0]
            
            yield train_idx, test_idx

    

In [5]:
def custom_metric(X_val, y_val, estimator, labels, X_train, y_train, *args, **kwargs):
    y_pred = estimator.predict(X_val)
    
    temp_df = pd.DataFrame({
        "product_id": X_val["product_id"].values,
        "y_true": y_val,
        "y_pred": y_pred
    })
    
    grouped = temp_df.groupby("product_id")[["y_true", "y_pred"]].sum()
    total_true = grouped["y_true"].sum()
    
    if total_true == 0:
        return 0.0, {"total_error": 0.0}
    
    total_error = np.abs(grouped["y_pred"] - grouped["y_true"]).sum() / total_true
    return total_error, {"total_error": total_error}

In [6]:
test_date = 33
train_date = 33
train_df = df[df["date_id"] <= train_date]
test_df = df[df["date_id"] == test_date]
X_train = train_df.drop("target", axis=1)
y_train = train_df["target"]
X_test = test_df.drop("target", axis=1)
y_test = test_df["target"]
tscv = CustomTimeSeriesSplit(n_splits=4, gap=1)
#time_budget = 1*60*60
time_budget = 3600
automl_settings = {
    "time_budget": time_budget,
    "task": "regression",
    "metric": custom_metric,
    "n_jobs": -1,
    "eval_method": "cv",
    "split_type": tscv,
    "verbose": 3,
    "retrain_full": True,
    "estimator_list": ['lgbm', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'catboost']
}
automl = AutoML()

In [7]:
automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

[flaml.automl.logger: 07-15 10:22:09] {1728} INFO - task = regression
[flaml.automl.logger: 07-15 10:22:09] {1739} INFO - Evaluation method: cv
[flaml.automl.logger: 07-15 10:22:09] {1838} INFO - Minimizing error metric: customized metric
[flaml.automl.logger: 07-15 10:22:09] {1955} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'catboost']
[flaml.automl.logger: 07-15 10:22:09] {2258} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 07-15 10:22:16] {2393} INFO - Estimated sufficient time budget=73744s. Estimated necessary time budget=616s.
[flaml.automl.logger: 07-15 10:22:16] {2442} INFO -  at 11.5s,	estimator lgbm's best error=1.0539,	best estimator lgbm's best error=1.0539
[flaml.automl.logger: 07-15 10:22:16] {2258} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 07-15 10:22:24] {2442} INFO -  at 19.1s,	estimator lgbm's best error=1.0539,	best estimator lgbm's best error=1.0539
[flaml.automl.logger: 07-15 1

In [8]:
# hago la prediccion en test
y_pred = automl.predict(X_test)
result = test_df[["product_id", "target"]].copy()
result["y_pred"] = y_pred
result = result.groupby("product_id").sum().reset_index()
result["error"] = np.abs(result["y_pred"] - result["target"])
total_error = result["error"].sum() / result["target"].sum()
print(f"Total error: {total_error}")
result

Total error: 0.09258218854665756


,product_id,target,y_pred,error
0,20001,1504.688599,1449.788330,54.900269
1,20002,1087.308594,1084.819824,2.488770
2,20003,892.501282,848.983643,43.517639
3,20004,637.900024,670.871094,32.971069
4,20005,593.244446,587.352966,5.891479
...,...,...,...,...
775,21263,0.012700,0.164697,0.151997
776,21265,0.050070,0.245678,0.195608
777,21266,0.051210,0.245678,0.194468
778,21267,0.015690,0.248813,0.233123


In [11]:
kaggle_date_id = 35
final_test_df = df[df["date_id"] == kaggle_date_id]

# reentreno el mejor modelo con todo el train
#best_model = automl.model
#best_model.fit(X=X_final_train, y=y_final_train)

y_pred_kaggle = automl.predict(final_test_df.drop("target", axis=1))
result_kaggle = final_test_df[["product_id", "date_id", "target"]].copy()
result_kaggle["y_pred"] = y_pred_kaggle
result_kaggle = result_kaggle.groupby("product_id").sum().reset_index()
submission = result_kaggle[["product_id", "y_pred"]].copy()
submission.columns = ["product_id", "tn"]
submission.to_csv(f"submission_automl_{time_budget}s.csv", index=False)
submission

,product_id,tn
0,20001,1504.566528
1,20002,1256.016479
2,20003,933.770203
3,20004,696.926392
4,20005,679.627991
...,...,...
775,21263,0.284209
776,21265,0.287332
777,21266,0.287332
778,21267,0.276655
